In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from harness.config import load_config
from harness.data import load_sample
from harness.models import load_model, run_classification
from harness.metrics import score_predictions

cfg = load_config(ROOT / "configs" / "default.yaml")
model_cfg, task = cfg["models"][0], cfg["tasks"][0]   # first model, first task

ds = load_sample(task["dataset"], task["split"], task["n_samples"], cfg["seed"])
tok, model = load_model(model_cfg["hf_id"])
rows, stats = run_classification(
    tok, model, ds, task["prompt"],
    text_key=task["text_key"], label_key=task["label_key"],
    detokenize=task["detokenize"], max_new_tokens=task["max_new_tokens"],
)

result = {"model": model_cfg["hf_id"], "task": task["name"], **score_predictions(rows), **stats}
print(pd.DataFrame([result]).to_string(index=False))